# detect_faces Colab 실행 확인 (A 이유노)

사전학습 가중치 yolov11n-face로 `src/detector.py`의 `detect_faces`·`reset_tracker`가 Colab GPU에서 돌아가는지 확인한다.

- 시작 전: **런타임 → 런타임 유형 변경 → T4 GPU**
- 사진·영상은 실행할 때 업로드한다. 얼굴 데이터는 저장소에 올리지 않는다.

In [ ]:
# 1. GPU 확인
!nvidia-smi -L

In [ ]:
# 2. 저장소 받기 (feat/detector가 main에 병합되면 BRANCH를 "main"으로)
BRANCH = "feat/detector"
!git clone -b {BRANCH} https://github.com/Leeyuno0419/Triple-Y-Mini.git
%cd Triple-Y-Mini

In [ ]:
# 3. 패키지 설치 (Colab에 있는 CUDA torch는 그대로 쓴다)
!pip install -q -r requirements.txt

In [ ]:
# 4. 사전학습 가중치 받기 → weights/best.pt
!wget -q -O weights/best.pt https://github.com/akanametov/yolo-face/releases/download/1.0.0/yolov11n-face.pt
!ls -lh weights

In [ ]:
# 5. 모델 불러오기 (import 시 한 번만 로드됨)
import cv2
import torch
from google.colab import files
from google.colab.patches import cv2_imshow

from src.detector import detect_faces, reset_tracker

print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "없음 (CPU)")

In [ ]:
# 6. 사진 탐지: 사진을 업로드하면 박스를 그려 보여준다
uploaded = files.upload()
img_path = next(iter(uploaded))
img = cv2.imread(img_path)  # BGR

boxes = detect_faces(img)
print("얼굴 수:", len(boxes))
print(boxes[:3])

vis = img.copy()
for b in boxes:
    x1, y1, x2, y2 = b["xyxy"]
    cv2.rectangle(vis, (x1, y1), (x2, y2), (0, 255, 0), 2)
cv2_imshow(vis)

In [ ]:
# 7. 영상 추적: mp4를 업로드하면 track_id를 그린 결과 영상을 만든다 (소리 없음, 확인용)
import time

uploaded = files.upload()
video_path = next(iter(uploaded))

cap = cv2.VideoCapture(video_path)
fps = cap.get(cv2.CAP_PROP_FPS) or 30
w, h = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
out = cv2.VideoWriter("track_result.mp4", cv2.VideoWriter_fourcc(*"mp4v"), fps, (w, h))

reset_tracker()  # 새 영상마다 처음에 한 번
ids, frames, start = set(), 0, time.time()
while True:
    ok, frame = cap.read()
    if not ok:
        break
    for b in detect_faces(frame, track=True):
        x1, y1, x2, y2 = b["xyxy"]
        cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
        if b["track_id"] is not None:
            ids.add(b["track_id"])
            cv2.putText(frame, str(b["track_id"]), (x1, y1 - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
    out.write(frame)
    frames += 1
cap.release()
out.release()

sec = time.time() - start
print(f"프레임 {frames}, 등장 인원(고유 track_id) {len(ids)}, 처리 {frames / sec:.1f} fps (읽기·그리기·저장 포함)")

In [ ]:
# 8. 결과 영상 내려받기
files.download("track_result.mp4")